# Procurement lab

Follow the six steps in the README. This notebook supports **step 2** (CSV, SQL and semantic layer) and **step 6** (SPARQL and workflow). Run section A, stop for steps 3–5, then return to section B. Keep the repository files together; open this notebook from the repository root or its notebooks folder. In Deepnote, keep data/ and db/ together under /work or the imported repository folder.

**Learning focus:** understand the ontology, mappings, SPARQL and workflow decisions. The Python code is provided: run the cells in order; no Python implementation or changes to worker functions are required.

## A. Data and semantic layer — step 2

### A1. Prepare the notebook

In [ ]:
%pip install pandas

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
from IPython.display import display

# Locate the lab files locally or in Deepnote's /work directory.
candidates = (Path.cwd(), Path.cwd().parent, Path("/work"))
ROOT = next((path for path in candidates
             if (path / "db/schema.sql").is_file()
             and (path / "data/departments.csv").is_file()), Path.cwd())

for relative_path in ("db/schema.sql", "data/departments.csv"):
    file_path = ROOT / relative_path
    assert file_path.is_file(), f"File not accessible to the Python kernel: {file_path}"
print(f"Lab directory: {ROOT}")

### A2. Inspect the source tables

Inspect the order IDs, amounts, statuses and foreign keys. Order 1006 is pending; order 1004 is rejected and has no positive approver.

In [ ]:
tables = {}

tables["departments"] = pd.read_csv(ROOT / "data/departments.csv")
display(tables["departments"])

In [ ]:
tables["employees"] = pd.read_csv(ROOT / "data/employees.csv")
display(tables["employees"])

In [ ]:
tables["suppliers"] = pd.read_csv(ROOT / "data/suppliers.csv")
display(tables["suppliers"])

In [ ]:
tables["purchase_orders"] = pd.read_csv(ROOT / "data/purchase_orders.csv")
display(tables["purchase_orders"])

### A3. Create the relational schema and load the records

Only the CREATE TABLE part of schema.sql is used: SQLite needs no CREATE DATABASE statement. Re-running this cell resets the local database. It does not change CSV files or Stardog.

In [ ]:
sql = sqlite3.connect(":memory:")
sql.execute("PRAGMA foreign_keys = ON")
schema = (ROOT / "db/schema.sql").read_text()
sql.executescript(schema[schema.index("CREATE TABLE departments"):])

for name, table in tables.items():
    table.to_sql(name, sql, if_exists="append", index=False)
assert not sql.execute("PRAGMA foreign_key_check").fetchall()

### Display helper

This cell only formats SQL results as a table; it contains no business logic.

In [ ]:
def show_query(query):
    display(pd.read_sql_query(query, sql))

### A4. Inspect the semantic layer

The view defines order counts and approved order value by department and currency. Read db/analytical_views.sql before running it. The Designer project supplies the complementary mappings of individual records to business entities; the aggregated view does not replace those mappings.

In [ ]:
sql.execute("DROP VIEW IF EXISTS procurement_metrics")
sql.executescript((ROOT / "db/analytical_views.sql").read_text())

show_query("""
SELECT department_name, currency,
       order_count, pending_order_count, approved_order_value
FROM procurement_metrics
ORDER BY department_id, currency
""")

**Check:** Procurement/CHF: 2 orders, 1 pending, approved value 12,500.00. Engineering/CHF: 2 orders, 0 pending, approved value 10,849.99. Why does a rejected order count as an order but contribute no approved value? Why are currencies separated?

**Stop here.** Complete README steps 3–5 in Designer, Explorer and Studio. The SQL database above is independent of Stardog.

## B. SPARQL and workflow — step 6

Use database `procurement` after completing README steps 3–5. **Your tasks:** enter the connection details, test the SPARQL query once, run the prepared worker, and observe the workflow. The Python adapter is provided and does not need to be edited or analyzed.

### B1. Install the Python clients

In [ ]:
%pip install martinlab-utils==0.1.14 pystardog==0.21.0

### B2. Connect to Stardog with Basic Authentication

First create a database user in **Studio → Security → Users → Add User**, then select that user and grant **Add Explicit Permission → READ → db → procurement**. Use the new database username and password below; your cloud portal login is separate. Enter the Stardog endpoint used by your Studio connection. See README step 6 for the setup steps.

The password is entered without echo; the prompts are cleared after entry. If hidden input is unavailable, the cell stops instead of falling back to visible password entry.


In [ ]:
import getpass
import warnings
import stardog

with warnings.catch_warnings():
    warnings.simplefilter("error", getpass.GetPassWarning)
    connection_details = {
        "endpoint": input("Stardog endpoint URL: ").strip(),
        "username": input("Stardog database user: ").strip(),
        "password": getpass.getpass("Stardog database password: "),
    }


### B3. Understand the SPARQL decision query

The query asks whether an order is classified as HighValueOrder by reasoning. It contains no fixed order number or threshold. Identify the business concepts and the EXISTS expression. The test supplies an example ID; later the workflow supplies its own orderId.


In [ ]:
CHECK_ORDER = """
PREFIX proc: <http://example.org/procurement#>

SELECT ?orderId ?amount ?status ?approvalRequired
WHERE {
    ?order a proc:PurchaseOrder ;
           proc:orderId ?orderId ;
           proc:amount ?amount ;
           proc:status ?status .
    FILTER(isNumeric(?amount) && ?amount >= 0)
    FILTER(STR(?status) IN ("pending", "approved"))
    BIND(EXISTS { ?order a proc:HighValueOrder } AS ?approvalRequired)
}
"""

### B4. Load the provided query adapter

Run this prepared cell unchanged. It executes the SPARQL query for the orderId supplied by a test or workflow instance and returns simple process variables. No Python implementation is required.


In [ ]:
from math import isfinite


def check_order(order_id):
    if type(order_id) is not int or order_id <= 0:
        raise ValueError("orderId must be a positive integer")
    with stardog.Connection("procurement", **connection_details) as conn:
        rows = conn.select(
            CHECK_ORDER, reasoning=True,
            bindings={"orderId": str(order_id)},
            default_graph_uri=["tag:stardog:api:context:all"]
        )["results"]["bindings"]
    if len(rows) != 1:
        raise ValueError(
            f"Order {order_id}: expected one eligible result, got {len(rows)}. "
            "Check the data graph selection, order ID, amount and status."
        )
    row = rows[0]
    amount = float(row["amount"]["value"])
    if not isfinite(amount):
        raise ValueError("Amount must be finite")
    return {"orderId": order_id, "amount": amount,
            "approvalRequired": row["approvalRequired"]["value"] in ("true", "1")}

### B5. Test one order

Use the same function that the workflow will call. The result is a simple table: order 1006 has amount 15250 and approvalRequired=True. You may change the example ID to inspect another order.


In [ ]:
import pandas as pd

pd.DataFrame([check_order(1006)])

### B6. Load the provided workflow adapter

Run this prepared cell unchanged. It passes the query result to the BPMN process and reports execution errors. The approval policy remains in Stardog.


In [ ]:
def determine_approval(taskid, response_data, worker):
    try:
        order_id = response_data[0]["variables"]["orderId"]["value"]
        result = check_order(order_id)
    except Exception as exc:
        worker.failure(taskid, error_message=str(exc), retries=0)
        return
    worker.complete(taskid, **result)


def auto_approve(taskid, response_data, worker):
    try:
        order_id = response_data[0]["variables"]["orderId"]["value"]
        result = check_order(order_id)
        if result["approvalRequired"]:
            raise ValueError("This order requires manager approval")
    except Exception as exc:
        worker.failure(taskid, error_message=str(exc), retries=0)
        return
    worker.complete(taskid, **result, orderStatus="approved")

### B7. Start the worker and process

Deploy the BPMN model first. The REST URL is fixed below. Enter your tenant ID when prompted; leave it empty only for a process without a tenant. Run this cell, then start a process with integer orderId=1003 for the automatic path or orderId=1006 for the manager path. Interrupt the kernel to stop polling before rerunning setup cells.

In [ ]:
import martinlab.camunda as cam

CAMUNDA_ENGINE_REST = "https://digibp.engines.aisl.science/engine-rest"
tenant_id = input("CIB seven tenant ID (empty = no tenant): ").strip() or None

worker = cam.Client(CAMUNDA_ENGINE_REST)
worker.subscribe("validate-order", determine_approval, tenant_id)
worker.subscribe("auto-approve", auto_approve, tenant_id)
worker.polling()

### B8. Observe the effect of changing a rule

Keep the B7 worker polling. B8 is an exercise in Studio and the process application; no additional Python cell is needed.

1. With the baseline threshold of 10,000, start a process instance with integer `orderId=1003`: it takes the automatic path.
2. In **Studio → Workspace**, select database `procurement` and graph `urn:Procurement_Lab:Procurement`, turn reasoning off, and execute the **SPARQL Update from README step 5**, with `FILTER(?amount > 4000)`. This replaces the stored rule content; leave `procurement-rules.ttl` unchanged.
3. Verify that the model graph contains one rule text with threshold 4000, using the query in README step 5. Switch the graph dropdown back to `stardog:context:all` and re-run the classification query with reasoning enabled: order 1003 now appears among the HighValueOrders.
4. Start a **new process instance** with integer `orderId=1003`: it now takes the manager path. No Python, worker restart or BPMN change is needed. The earlier instance is unaffected.
5. In the same SPARQL Update, change `4000` to `10000` and execute it again to restore the baseline. Interrupt notebook polling before running further Python cells. The provided Python cells remain unchanged throughout the exercise.

The worker demonstrates routing: automatic approval sets only the process variable `orderStatus`. Neither automatic nor human decisions are written back to Stardog or CSV. Runtime errors become incidents.
